In [1]:
import sys
!"{sys.executable}" -m pip install shap


[notice] A new release of pip is available: 23.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# CASE 8: Feature Selection Impact Study
# CICIDS2017 — Ensemble ML Intrusion Detection
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os

warnings.filterwarnings('ignore')

# ML models
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Feature selection
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, classification_report)

# Split
from sklearn.model_selection import train_test_split

# SHAP
import shap

print(" All libraries imported successfully")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

 All libraries imported successfully
NumPy: 2.2.5
Pandas: 2.3.3


In [3]:
# ============================================================
# LOAD DATASET
# ============================================================
# Update this path to wherever your preprocessed CSV is stored
DATA_PATH = "CICIDS2017_preprocessed.csv"

print(" Loading dataset...")
df = pd.read_csv(DATA_PATH)

print(f" Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nLabel distribution:")
print(df['Label'].value_counts().sort_index())

 Loading dataset...
 Dataset loaded: 756,208 rows × 67 columns

Label distribution:
Label
0     628486
1        584
2      38404
3       3086
4      51854
5       1568
6       1616
7       1779
8          3
9         11
10     27208
11       966
12       441
13         6
14       196
Name: count, dtype: int64


In [4]:
# ============================================================
# PREPARE X, y — SAME SPLIT AS ALL OTHER CASES
# ============================================================

# Separate features and labels
X = df.drop(columns=['Label'])
y = df['Label']

print(f"Feature matrix shape: {X.shape}")
print(f"Number of classes: {y.nunique()}")
print(f"Classes: {sorted(y.unique())}")


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y           # preserves class ratios despite imbalance
)

print(f"\n Train size: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f" Test size:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nBaseline feature count: {X_train.shape[1]}")

Feature matrix shape: (756208, 66)
Number of classes: 15
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]

 Train size: 529,345 samples (70.0%)
 Test size:  226,863 samples (30.0%)

Baseline feature count: 66


In [5]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name, strategy_name, n_features):
    """Train a model and return all metrics as a dict."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    
    y_pred = model.predict(X_te)
    
    return {
        'Strategy':          strategy_name,
        'Model':             model_name,
        'N_Features':        n_features,
        'Accuracy':          round(accuracy_score(y_te, y_pred) * 100, 4),
        'Precision (Macro)': round(precision_score(y_te, y_pred, average='macro', zero_division=0) * 100, 4),
        'Recall (Macro)':    round(recall_score(y_te, y_pred, average='macro', zero_division=0) * 100, 4),
        'F1-Macro':          round(f1_score(y_te, y_pred, average='macro', zero_division=0) * 100, 4),
        'F1-Weighted':       round(f1_score(y_te, y_pred, average='weighted', zero_division=0) * 100, 4),
        'MCC':               round(matthews_corrcoef(y_te, y_pred), 4),
        'Train Time (s)':    round(train_time, 2)
    }


def get_models():
    """Return the 3 models to test for each strategy."""
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    xgb = XGBClassifier(n_estimators=100, n_jobs=-1, random_state=42,
                        use_label_encoder=False, eval_metric='mlogloss',
                        tree_method='hist')    # faster training
    
    stacked = StackingClassifier(
        estimators=[
            ('dt',  DecisionTreeClassifier(random_state=42)),
            ('rf',  RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=50, n_jobs=-1, random_state=42,
                                  use_label_encoder=False, eval_metric='mlogloss',
                                  tree_method='hist'))
        ],
        final_estimator=LogisticRegression(max_iter=1000, n_jobs=-1),
        n_jobs=-1
    )
    
    return [('Random Forest', rf), ('XGBoost', xgb), ('Stacked Ensemble', stacked)]


def run_strategy(strategy_name, X_tr, X_te, y_tr, y_te, n_features):
    """Run all 3 models on the given feature-selected data."""
    print(f"\n{'='*60}")
    print(f"  Strategy: {strategy_name}  |  Features: {n_features}")
    print(f"{'='*60}")
    
    results = []
    for model_name, model in get_models():
        print(f"  ▶ Training {model_name}...", end=' ', flush=True)
        r = evaluate_model(model, X_tr, X_te, y_tr, y_te,
                           model_name, strategy_name, n_features)
        results.append(r)
        print(f"Done   Acc={r['Accuracy']}%  F1-Macro={r['F1-Macro']}%  MCC={r['MCC']}")
    
    return results


# Storage for all results
all_results = []
print(" Helper functions defined")

 Helper functions defined


In [7]:
# ============================================================
# STRATEGY 1: NO FEATURE SELECTION — All 66 features
# ============================================================
# Theory: This is our within-case baseline. We use all 66
# features so we know exactly what feature selection adds/removes.
# Compare this directly against Case 1 RF/XGB/Stacked results.

print(" STRATEGY 1: No Feature Selection (Baseline)")

results_s1 = run_strategy(
    strategy_name='S1_No_Selection',
    X_tr=X_train, X_te=X_test,
    y_tr=y_train, y_te=y_test,
    n_features=X_train.shape[1]
)

all_results.extend(results_s1)

# Save checkpoint
pd.DataFrame(all_results).to_csv('case8_results_checkpoint.csv', index=False)
print("\n Checkpoint saved.")

 STRATEGY 1: No Feature Selection (Baseline)

  Strategy: S1_No_Selection  |  Features: 66
  ▶ Training Random Forest... Done   Acc=99.8166%  F1-Macro=79.215%  MCC=0.9939
  ▶ Training XGBoost... Done   Acc=99.2454%  F1-Macro=54.384%  MCC=0.975
  ▶ Training Stacked Ensemble... Done   Acc=99.8316%  F1-Macro=71.867%  MCC=0.9944

 Checkpoint saved.


In [ ]:
# ============================================================
# STRATEGY 2: CORRELATION-BASED SELECTION (Pearson r > 0.95)
# ============================================================
# Theory: When two features have |r| > 0.95, they contain nearly
# identical information. Keeping both wastes computation and can
# cause multicollinearity problems (especially for Logistic
# Regression in the stacking meta-learner). We drop one from
# each highly correlated pair.

print(" STRATEGY 2: Correlation-Based Feature Selection")

# Compute correlation matrix on training data ONLY (avoid data leakage)
corr_matrix = X_train.corr(method='pearson').abs()

# Find upper triangle pairs with r > 0.95
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f"Features removed (high correlation): {len(to_drop_corr)}")
print(f"Features removed: {to_drop_corr}")
print(f"Remaining features: {X_train.shape[1] - len(to_drop_corr)}")

X_train_corr = X_train.drop(columns=to_drop_corr)
X_test_corr  = X_test.drop(columns=to_drop_corr)

results_s2 = run_strategy(
    strategy_name='S2_Correlation',
    X_tr=X_train_corr, X_te=X_test_corr,
    y_tr=y_train, y_te=y_test,
    n_features=X_train_corr.shape[1]
)

all_results.extend(results_s2)
pd.DataFrame(all_results).to_csv('case8_results_checkpoint.csv', index=False)
print("\n Checkpoint saved.")

In [ ]:
# ============================================================
# STRATEGY 3: MUTUAL INFORMATION BASED SELECTION
# ============================================================
# Theory: Mutual Information (MI) measures how much knowing
# a feature reduces uncertainty about the label. Unlike Pearson
# correlation, MI captures non-linear relationships — critical
# for attack traffic that has complex patterns.
#
# Formula: MI(X;Y) = Σ p(x,y) * log[p(x,y) / (p(x)*p(y))]
#
# We test top 30 features (≈45% of 66). This is a common
# sweet spot — enough signal, less noise.

print(" STRATEGY 3: Mutual Information Based Selection")

print("  Computing Mutual Information scores (may take 1-2 min)...")
# Use a sample of training data for speed (MI on 529K rows is slow)
# Sample 100K rows — still very representative
SAMPLE_SIZE = 100_000
idx = np.random.RandomState(42).choice(len(X_train), SAMPLE_SIZE, replace=False)
X_sample = X_train.iloc[idx]
y_sample = y_train.iloc[idx]

mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42, n_jobs=-1)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

TOP_N_MI = 30
top_mi_features = mi_series.head(TOP_N_MI).index.tolist()
print(f"  Top {TOP_N_MI} features selected")
print(f"  Top 10 by MI: {top_mi_features[:10]}")

X_train_mi = X_train[top_mi_features]
X_test_mi  = X_test[top_mi_features]

results_s3 = run_strategy(
    strategy_name=f'S3_MutualInfo_Top{TOP_N_MI}',
    X_tr=X_train_mi, X_te=X_test_mi,
    y_tr=y_train, y_te=y_test,
    n_features=TOP_N_MI
)

all_results.extend(results_s3)
pd.DataFrame(all_results).to_csv('case8_results_checkpoint.csv', index=False)
print("\n Checkpoint saved.")

# Plot MI scores
plt.figure(figsize=(12, 5))
mi_series.head(30).plot(kind='bar', color='steelblue')
plt.title('Top 30 Features by Mutual Information Score', fontsize=14)
plt.xlabel('Feature')
plt.ylabel('MI Score')
plt.tight_layout()
plt.savefig('case8_mutual_information_scores.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STRATEGY 4: RANDOM FOREST IMPORTANCE BASED SELECTION
# ============================================================
# Theory: Random Forests compute "feature importance" as the
# average decrease in node impurity (Gini impurity) caused by
# each feature across all trees. Features that appear in splits
# near the top of trees (affecting many samples) get high scores.
# This is very fast and works well for tree-based final models.

print(" STRATEGY 4: Random Forest Importance Based Selection")

print("  Training selector RF (100 trees)...")
selector_rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
selector_rf.fit(X_train, y_train)

rf_importances = pd.Series(selector_rf.feature_importances_,
                            index=X_train.columns).sort_values(ascending=False)

TOP_N_RF = 30
top_rf_features = rf_importances.head(TOP_N_RF).index.tolist()
print(f"  Top {TOP_N_RF} features selected")
print(f"  Top 10 by RF importance: {top_rf_features[:10]}")

X_train_rf = X_train[top_rf_features]
X_test_rf  = X_test[top_rf_features]

results_s4 = run_strategy(
    strategy_name=f'S4_RFImportance_Top{TOP_N_RF}',
    X_tr=X_train_rf, X_te=X_test_rf,
    y_tr=y_train, y_te=y_test,
    n_features=TOP_N_RF
)

all_results.extend(results_s4)
pd.DataFrame(all_results).to_csv('case8_results_checkpoint.csv', index=False)
print("\n Checkpoint saved.")

# Plot RF importances
plt.figure(figsize=(12, 5))
rf_importances.head(30).plot(kind='bar', color='darkorange')
plt.title('Top 30 Features by Random Forest Importance', fontsize=14)
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('case8_rf_importance_scores.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STRATEGY 5: SHAP-Based Feature Selection
# ============================================================

print(" STRATEGY 5: SHAP-Based Feature Selection")

print("  Computing SHAP values on 5,000 test samples (may take 2-3 min)...")

SHAP_SAMPLE = 5000
idx_shap = np.random.RandomState(42).choice(len(X_test), SHAP_SAMPLE, replace=False)
X_shap_sample = X_test.iloc[idx_shap]

explainer = shap.TreeExplainer(selector_rf)
shap_values = explainer.shap_values(X_shap_sample)

# ── Handle both SHAP output formats ──────────────────────────
# Old SHAP: list of arrays, one per class → shape: [(n_samples, n_features), ...]
# New SHAP: single 3D array               → shape: (n_samples, n_features, n_classes)

if isinstance(shap_values, list):
    # Old format: list of (n_samples, n_features) arrays
    mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    # New format: (n_samples, n_features, n_classes)
    mean_shap = np.abs(shap_values).mean(axis=(0, 2))  # avg over samples & classes
else:
    # 2D binary case: (n_samples, n_features)
    mean_shap = np.abs(shap_values).mean(axis=0)

print(f"  mean_shap shape: {mean_shap.shape}  |  n_features: {X_train.shape[1]}")

shap_series = pd.Series(mean_shap, index=X_train.columns).sort_values(ascending=False)

TOP_N_SHAP = 30
top_shap_features = shap_series.head(TOP_N_SHAP).index.tolist()
print(f"  Top {TOP_N_SHAP} features selected")
print(f"  Top 10 by SHAP: {top_shap_features[:10]}")

X_train_shap = X_train[top_shap_features]
X_test_shap  = X_test[top_shap_features]

results_s5 = run_strategy(
    strategy_name=f'S5_SHAP_Top{TOP_N_SHAP}',
    X_tr=X_train_shap, X_te=X_test_shap,
    y_tr=y_train, y_te=y_test,
    n_features=TOP_N_SHAP
)

all_results.extend(results_s5)
pd.DataFrame(all_results).to_csv('case8_results_checkpoint.csv', index=False)
print("\n Checkpoint saved.")

# Plot SHAP scores
plt.figure(figsize=(12, 5))
shap_series.head(30).plot(kind='bar', color='mediumseagreen')
plt.title('Top 30 Features by Mean |SHAP| Value', fontsize=14)
plt.xlabel('Feature')
plt.ylabel('Mean |SHAP|')
plt.tight_layout()
plt.savefig('case8_shap_scores.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# COMPILE ALL RESULTS
# ============================================================

results_df = pd.DataFrame(all_results)

# Save final CSV
results_df.to_csv('Case8_Results.csv', index=False)
print(" Final results saved to Case8_Results.csv")
print(f"\nTotal experiments: {len(results_df)}")
print("\n", results_df.to_string(index=False))

In [ ]:
# ============================================================
# COMPARE AGAINST CASE 1 BASELINE
# ============================================================

case1_baseline = pd.DataFrame([
    {'Model': 'Random Forest',    'Accuracy': 99.81, 'F1-Macro': 79.29, 'MCC': 0.9937},
    {'Model': 'XGBoost',          'Accuracy': 99.25, 'F1-Macro': 54.38, 'MCC': 0.9750},
    {'Model': 'Stacked Ensemble', 'Accuracy': 99.83, 'F1-Macro': 71.83, 'MCC': 0.9943},
])

print("=" * 70)
print("  CASE 1 BASELINE (all 78 features, no normalization)")
print("=" * 70)
print(case1_baseline.to_string(index=False))

print("\n" + "=" * 70)
print("  CASE 8 RESULTS BY STRATEGY")
print("=" * 70)

# Best result per strategy × model
pivot = results_df.pivot_table(
    index=['Strategy', 'N_Features'],
    columns='Model',
    values=['Accuracy', 'F1-Macro', 'MCC'],
    aggfunc='first'
)
print(pivot.to_string())

# Which strategy gave highest F1-Macro per model?
print("\n" + "=" * 70)
print("  BEST STRATEGY PER MODEL (by F1-Macro)")
print("=" * 70)
best = results_df.loc[results_df.groupby('Model')['F1-Macro'].idxmax()]
print(best[['Model', 'Strategy', 'N_Features', 'Accuracy', 'F1-Macro', 'MCC']].to_string(index=False))

In [ ]:
# ============================================================
# VISUALIZATION: F1-Macro by Strategy × Model
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
models = ['Random Forest', 'XGBoost', 'Stacked Ensemble']
case1_f1 = {'Random Forest': 79.29, 'XGBoost': 54.38, 'Stacked Ensemble': 71.83}
colors = plt.cm.Set2(np.linspace(0, 1, results_df['Strategy'].nunique()))

for ax, model in zip(axes, models):
    model_df = results_df[results_df['Model'] == model].copy()
    short_labels = [s.split('_')[0] + '_' + s.split('_')[1] for s in model_df['Strategy']]
    
    bars = ax.bar(short_labels, model_df['F1-Macro'], color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(y=case1_f1[model], color='red', linestyle='--', linewidth=2,
               label=f'Case1 baseline ({case1_f1[model]}%)')
    
    for bar, val in zip(bars, model_df['F1-Macro']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
    
    ax.set_title(f'{model}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature Selection Strategy')
    ax.set_ylabel('F1-Macro (%)')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(model_df['F1-Macro'].max(), case1_f1[model]) + 10)

plt.suptitle('Case 8: F1-Macro by Feature Selection Strategy\n(red dashed = Case 1 baseline)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('case8_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Feature count vs F1-Macro scatter ──────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for model, group in results_df.groupby('Model'):
    ax.scatter(group['N_Features'], group['F1-Macro'], label=model, s=100, zorder=3)
    for _, row in group.iterrows():
        ax.annotate(row['Strategy'].split('_')[1],
                    (row['N_Features'], row['F1-Macro']),
                    textcoords='offset points', xytext=(5, 3), fontsize=8)

ax.set_xlabel('Number of Features Used', fontsize=12)
ax.set_ylabel('F1-Macro (%)', fontsize=12)
ax.set_title('Features Used vs F1-Macro Performance', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('case8_features_vs_f1.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# FEATURE OVERLAP: Which features are consistently important?
# ============================================================

# Compare which features appear in MI, RF, and SHAP top lists
mi_set   = set(top_mi_features)
rf_set   = set(top_rf_features)
shap_set = set(top_shap_features)

# Features selected by ALL 3 methods = most reliable important features
consensus_features = mi_set & rf_set & shap_set
print(f"Features selected by ALL 3 methods (consensus): {len(consensus_features)}")
print(sorted(consensus_features))

# Venn-style summary
print(f"\nMI ∩ RF: {len(mi_set & rf_set)} features")
print(f"MI ∩ SHAP: {len(mi_set & shap_set)} features")
print(f"RF ∩ SHAP: {len(rf_set & shap_set)} features")
print(f"MI ∩ RF ∩ SHAP: {len(consensus_features)} features")

# Heatmap: feature × method (binary — selected or not)
all_methods_features = sorted(mi_set | rf_set | shap_set)
heatmap_df = pd.DataFrame({
    'Mutual Info':    [1 if f in mi_set else 0 for f in all_methods_features],
    'RF Importance':  [1 if f in rf_set else 0 for f in all_methods_features],
    'SHAP':           [1 if f in shap_set else 0 for f in all_methods_features],
}, index=all_methods_features)

plt.figure(figsize=(8, max(8, len(all_methods_features)//3)))
sns.heatmap(heatmap_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, cbar=False)
plt.title('Feature Selection Overlap Across Methods\n(1 = selected, 0 = not selected)',
          fontsize=13)
plt.tight_layout()
plt.savefig('case8_feature_overlap_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================


print("CASE 8 FINAL SUMMARY")


summary = results_df.groupby('Strategy').agg(
    N_Features=('N_Features', 'first'),
    Avg_Accuracy=('Accuracy', 'mean'),
    Avg_F1_Macro=('F1-Macro', 'mean'),
    Best_F1_Macro=('F1-Macro', 'max'),
    Avg_MCC=('MCC', 'mean'),
    Avg_TrainTime=('Train Time (s)', 'mean')
).round(4)

print(summary.to_string())
print()

best_strategy = summary['Avg_F1_Macro'].idxmax()
best_f1 = summary['Avg_F1_Macro'].max()
print(f"BEST STRATEGY (by avg F1-Macro): {best_strategy}")
print(f"   Average F1-Macro: {best_f1:.2f}%")
print()

# Answering the research questions
print("=" * 60)
print("RESEARCH QUESTIONS ANSWERED:")
print("=" * 60)

# Compare S1 (no selection) vs best
s1_avg_f1 = results_df[results_df['Strategy']=='S1_No_Selection']['F1-Macro'].mean()
best_avg_f1 = results_df[results_df['Strategy']==best_strategy]['F1-Macro'].mean()
delta = best_avg_f1 - s1_avg_f1

print(f"\n1. Does reducing features improve performance?")
print(f"   Baseline (S1, 66 features) avg F1-Macro: {s1_avg_f1:.2f}%")
print(f"   Best strategy avg F1-Macro: {best_avg_f1:.2f}%")
print(f"   Δ = {delta:+.2f}% → {'IMPROVED ' if delta > 0 else 'HURT ' if delta < -0.5 else 'SIMILAR '}")

print(f"\n2. Which feature selection method is best?")
print(f"   → {best_strategy}")

print(f"\n3. Which features matter most?")
print(f"   Consensus (MI ∩ RF ∩ SHAP): {sorted(consensus_features)[:10]}")

print(f"\n4. Case 1 comparison (RF, no feature selection):")
print(f"   Case 1 RF: F1-Macro=79.29%, MCC=0.9937")
case8_rf_best = results_df[results_df['Model']=='Random Forest']['F1-Macro'].max()
case8_rf_best_row = results_df[results_df['Model']=='Random Forest'].loc[
    results_df[results_df['Model']=='Random Forest']['F1-Macro'].idxmax()]
print(f"   Case 8 RF best: F1-Macro={case8_rf_best:.2f}% "
      f"({case8_rf_best_row['Strategy']}, {case8_rf_best_row['N_Features']} features)")